# Importing Necessary Libraries

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, GroupKFold
from lightgbm import LGBMClassifier, LGBMRanker

from scipy.sparse import hstack
from tqdm import tqdm
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
train_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [3]:
print("Dataset preview : \n")
train_data.head()

Dataset preview : 



,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [4]:
print(f"Train data size : {train_data.shape}")
print(f"Test data size : {test_data.shape}")

Train data size : (2000, 8)
Test data size : (500, 7)


# Preprocessing

In [5]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = text.replace("\n"," ")
    text = " ".join(text.split())
    return text

In [6]:
for col in ["prompt","A","B","C","D","E"]:
    train_data[col] = train_data[col].fillna("").apply(clean_text)
    test_data[col] = test_data[col].fillna("").apply(clean_text)

# Convert MCQ into Ranking Format

In [7]:
options = ["A","B","C","D","E"]

train_rows = []

for _, row in tqdm(train_data.iterrows(), total=len(train_data)):
    
    correct = row["answer"]
    
    for op in options:
        
        train_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "option": row[op],
            "option_label": op,
            "label": int(op == correct)
        })

train_long = pd.DataFrame(train_rows)

print(train_long.shape)

100%|██████████| 2000/2000 [00:00<00:00, 12849.45it/s]

(10000, 5)


# Test Ranking Format

In [8]:
test_rows = []

for _, row in tqdm(test_data.iterrows(), total=len(test_data)):
    
    for op in options:
        
        test_rows.append({
            "id": row["id"],
            "option_label": op,
            "prompt": row["prompt"],
            "option": row[op]
        })

test_long = pd.DataFrame(test_rows)

100%|██████████| 500/500 [00:00<00:00, 14242.60it/s]


# Create Additional Features

## Length Features

In [9]:
for df in [train_long, test_long]:
    df["prompt_len"] = df["prompt"].str.len()
    df["option_len"] = df["option"].str.len()
    df["prompt_words"] = df["prompt"].str.split().str.len()
    df["option_words"] = df["option"].str.split().str.len()

In [10]:
num_cols = [
    "prompt_len",
    "option_len",
    "prompt_words",
    "option_words"
]

## Word Overlap and Similarity Feature

In [11]:
def overlap_count(q, a):
    
    q_words = set(q.split())
    a_words = set(a.split())
    
    return len(q_words & a_words)
def overlap_ratio(q, a):

    q_words = set(q.split())
    a_words = set(a.split())

    if len(q_words) == 0:
        return 0

    return len(q_words & a_words) / len(q_words)

In [12]:
for df in [train_long, test_long]:

    df["overlap"] = [
        overlap_count(q, a)
        for q, a in zip(df["prompt"], df["option"])
    ]
    df["overlap_ratio"] = [
    overlap_ratio(q, a)
    for q, a in zip(df["prompt"], df["option"])
    ]

# TF-IDF Feature Engineering

In [13]:
train_text = (
    train_long["prompt"] +
    " [SEP] " +
    train_long["option"]
)

test_text = (
    test_long["prompt"] +
    " [SEP] " +
    test_long["option"]
)

In [14]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    stop_words="english"
)

In [15]:
char_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,6),
    min_df=2,
    max_features=30000
)

In [16]:
X_char_train = char_vec.fit_transform(train_text)
X_char_test = char_vec.transform(test_text)

In [17]:
X_word_train = tfidf.fit_transform(train_text)

X_word_test = tfidf.transform(test_text)

## Add Cosine Similarity

In [18]:
train_prompt_vec = tfidf.transform(train_long["prompt"])
train_option_vec = tfidf.transform(train_long["option"])

test_prompt_vec = tfidf.transform(test_long["prompt"])
test_option_vec = tfidf.transform(test_long["option"])

In [19]:
train_long["cosine_sim"] = [
    cosine_similarity(
        train_prompt_vec[i],
        train_option_vec[i]
    )[0][0]
    for i in range(len(train_long))
]

test_long["cosine_sim"] = [
    cosine_similarity(
        test_prompt_vec[i],
        test_option_vec[i]
    )[0][0]
    for i in range(len(test_long))
]

In [20]:
num_cols.extend([
    "overlap",
    "overlap_ratio",
    "cosine_sim"
])

In [21]:
num_cols

['prompt_len',
 'option_len',
 'prompt_words',
 'option_words',
 'overlap',
 'overlap_ratio',
 'cosine_sim']

In [22]:
question_ids = train_long["id"]

In [23]:
X_num_train = csr_matrix(
    train_long[num_cols].values.astype(np.float32)
)

X_num_test = csr_matrix(
    test_long[num_cols].values.astype(np.float32)
)

In [24]:
X_train = hstack([
    X_word_train,
    X_char_train,
    X_num_train
]).tocsr()

X_test = hstack([
    X_word_test,
    X_char_test,
    X_num_test
]).tocsr()

In [25]:
print(type(X_train))
print(type(X_test))

<class 'scipy.sparse._csr.csr_matrix'>
<class 'scipy.sparse._csr.csr_matrix'>


In [26]:
print(X_word_train.shape)
print(X_char_train.shape)
print(X_num_train.shape)
print(X_train.shape)

(10000, 11244)
(10000, 30000)
(10000, 7)
(10000, 41251)


In [27]:
train_long[num_cols].isnull().sum()

prompt_len       0
option_len       0
prompt_words     0
option_words     0
overlap          0
overlap_ratio    0
cosine_sim       0
dtype: int64

In [28]:
y = train_long["label"]

# LightGBM

## LGBMClassifier

In [29]:
lgb_params = {
    "n_estimators":1500,
    "learning_rate":0.03,
    "num_leaves":127,
    "subsample":0.8,
    "colsample_bytree":0.8,
    "objective":"binary",
    "metric":"auc",
    "verbosity":-1,
    "random_state":42
}

## Train

In [30]:
# # skf = StratifiedKFold(
# #     n_splits=5,
# #     shuffle=True,
# #     random_state=42
# # )
# gkf = GroupKFold(n_splits=5)

# oof = np.zeros(len(train_long))
# test_preds = np.zeros(len(test_long))

# # for fold, (tr_idx, val_idx) in enumerate(
# #         skf.split(X_train, y)):
# groups = train_long["id"]

# for fold, (tr_idx, val_idx) in enumerate(
#     gkf.split(X_train, y, groups)
# ):
    
#     print(f"Fold {fold+1}")
    
#     model = LGBMClassifier(**lgb_params)
    
#     model.fit(
#         X_train[tr_idx],
#         y.iloc[tr_idx]
#     )
    
#     oof[val_idx] = model.predict_proba(
#         X_train[val_idx]
#     )[:,1]
    
#     test_preds += (
#         model.predict_proba(X_test)[:,1] / 5
#     )
#     print(test_preds)

## LGBMRanker

In [31]:
ranker_params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 63,
    "force_col_wise": True,
    "verbosity": -1,
    "random_state": 42,
    "n_jobs": -1
}

In [32]:

print(X_train.shape)
print(y.value_counts())

(10000, 41251)
label
0    8000
1    2000
Name: count, dtype: int64


## Training

In [33]:
gkf = GroupKFold(n_splits=5)

groups = train_long["id"]

oof = np.zeros(len(train_long))
test_preds = np.zeros(len(test_long))

for fold, (tr_idx, val_idx) in enumerate(
        gkf.split(X_train, y, groups)
):

    print(f"Fold {fold+1}")

    ranker = LGBMRanker(**ranker_params)

    # group sizes for training fold
    train_groups = (
        train_long.iloc[tr_idx]
        .groupby("id")
        .size()
        .values
    )

    valid_groups = (
        train_long.iloc[val_idx]
        .groupby("id")
        .size()
        .values
    )

    ranker.fit(
        X_train[tr_idx],
        y.iloc[tr_idx],
        group=train_groups,
        eval_set=[(X_train[val_idx], y.iloc[val_idx])],
        eval_group=[valid_groups]
    )

    oof[val_idx] = ranker.predict(
        X_train[val_idx]
    )

    test_preds += (
        ranker.predict(X_test) / 5
    )

Fold 1


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


Fold 2


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


Fold 3


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


Fold 4


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


Fold 5


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(


# Evaluation

In [34]:
def map3_score(df):

    scores = []

    for _, g in df.groupby("id"):

        pred = (
            g.sort_values(
                "oof_pred",
                ascending=False
            )["option_label"]
            .tolist()[:3]
        )

        true = g.loc[
            g["label"] == 1,
            "option_label"
        ].iloc[0]

        if true == pred[0]:
            scores.append(1.0)

        elif true == pred[1]:
            scores.append(0.5)

        elif true == pred[2]:
            scores.append(1/3)

        else:
            scores.append(0)

    return np.mean(scores)

In [35]:
train_long["oof_pred"] = oof

cv_score = map3_score(train_long)

print("CV MAP@3 =", cv_score)

CV MAP@3 = 0.99025


In [36]:
# for _, g in train_long.groupby("id"):
#     print(g[["option_label","label","oof_pred"]])
#     break

In [37]:
correct = 0

for _, g in train_long.groupby("id"):
    pred = g.loc[g["oof_pred"].idxmax(), "option_label"]
    true = g.loc[g["label"] == 1, "option_label"].iloc[0]

    if pred == true:
        correct += 1

print(correct / train_data.shape[0])

0.984


In [38]:
train_ids = set(train_long.iloc[tr_idx]["id"])
val_ids = set(train_long.iloc[val_idx]["id"])

print(len(train_ids & val_ids))

0


In [39]:
for _, g in train_long.groupby("id"):

    print(
        g[["option_label","label","oof_pred"]]
        .sort_values("oof_pred", ascending=False)
    )

    break

  option_label  label  oof_pred
1            B      1  5.999716
0            A      0 -3.996306
2            C      0 -4.475086
4            E      0 -5.470565
3            D      0 -5.802729


In [40]:
sample_ids = np.random.choice(
    train_long["id"].unique(),
    10,
    replace=False
)

for sid in sample_ids:

    g = train_long[train_long["id"] == sid]

    print("\n", sid)

    print(
        g[["option_label","label","oof_pred"]]
        .sort_values(
            "oof_pred",
            ascending=False
        )
    )


 7
   option_label  label  oof_pred
34            E      1  4.053508
31            B      0 -2.528755
33            D      0 -4.803016
30            A      0 -6.149231
32            C      0 -6.226191

 643
     option_label  label  oof_pred
3213            D      1  4.685763
3210            A      0 -3.720265
3212            C      0 -4.409674
3214            E      0 -4.966190
3211            B      0 -4.989746

 953
     option_label  label  oof_pred
4761            B      1  3.076099
4762            C      0 -5.553460
4763            D      0 -5.590881
4760            A      0 -6.015830
4764            E      0 -6.516888

 1195
     option_label  label  oof_pred
5971            B      1  5.048985
5973            D      0 -4.464470
5974            E      0 -5.377676
5972            C      0 -6.216946
5970            A      0 -7.009849

 1600
     option_label  label  oof_pred
7997            C      1  3.963750
7995            A      0 -2.799888
7998            D      0 -4.919328
79

## Predictions

In [41]:
test_long["score"] = test_preds

In [42]:
test_long.head()

,id,option_label,prompt,option,prompt_len,option_len,prompt_words,option_words,overlap,overlap_ratio,cosine_sim,score
0,1,A,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",144,117,19,17,1,0.058824,0.0,4.860742
1,1,B,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",144,117,19,17,0,0.000000,0.0,-4.757495
2,1,C,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",144,118,19,17,0,0.000000,0.0,-5.368433
3,1,D,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",144,120,19,17,0,0.000000,0.0,-5.332963
4,1,E,pick the best possible answer: what is the rel...,"for every eigenstate of one hamiltonian, its p...",144,116,19,17,0,0.000000,0.0,-4.745364


# Submission

In [43]:
submission_rows = []

for qid, group in test_long.groupby("id"):
    
    ranked = group.sort_values(
        "score",
        ascending=False
    )
    
    top3 = ranked["option_label"].head(3).tolist()
    
    submission_rows.append({
        "id": qid,
        "prediction": " ".join(top3)
    })

In [44]:
submission = pd.DataFrame(submission_rows)
submission.head()

,id,prediction
0,1,A E B
1,2,B D E
2,3,B D E
3,4,E C D
4,5,C D A


In [45]:
submission.to_csv(
    "submission.csv",
    index=False
)